<a href="https://colab.research.google.com/github/andrigerber/Adv_GenAI/blob/main/test_retrival_GA_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# step_2_main.ipynb

**Advanced Generative AI – Pre-processing Notebook**  
**Semester Project**

- **Supervisor:** Dr. Guang Lu  
- **Students:** Andri Gerber · Amir Shatrolli · Kened Kqiraj

---

## Overview 1-9

This notebook refines our pre-processing pipeline for **multilingual documents** (EN/DE/FR/IT) and ensures both `main_content` and paragraphs are cleaned using **the same logic**, all **lowercased**. We unify everything into a **single** function with a parameter to handle either a single text field or a list of paragraphs.

Ultimately, each JSON file under the specified `ROOT_DIR` is updated with:

1. A **human-readable title** (`title`) extracted from filenames (if needed).  
2. A **cleaned** main content (`main_content_clean`).  
3. A **cleaned** paragraph list (`paragraphs_cleaned_1`).  

We then apply **two chunking methods** (fixed-size and semantic) to prepare documents for retrieval tasks. This step yields:

- **`chunks_fixed_size`**: Divided by token-length (~512 tokens) with overlap (default 64).  
- **`chunks_semantic`**: Grouped by topic similarity (cosine-sim threshold), merged until each chunk is at least 50 tokens.  

Finally, we add **metadata** (named entities, keywords, etc.) to each chunk, then produce a final set of JSON records—one for doc-level and additional ones for each chunk. This pipeline is **idempotent**, so re-running it won’t harm existing data.

---

### **Table of Contents**

1. [Install Dependencies](#install-dependencies)  
2. [Mount Google Drive](#mount-google-drive)  
3. [One-Time Helpers & Utilities](#helpers)  
4. [Single Processing Function](#single-processing-function)  
5. [Compute Title from Filenames](#compute-title-from-filenames)  
6. [Run the Main Pre-processing Pipeline](#run-pipeline)  
7. [Sanity Check](#sanity-check)  
8. [Chunking (Fixed-Size & Semantic)](#chunking)  
9. [Metadata Enrichment & Final JSON Output](#metadata-and-final-json)  

---


## 1. Install Dependencies <a id="install-dependencies"></a>

Below, we install or update the necessary Python libraries for text cleaning and language handling.

In [ ]:
!pip -q install pandas tqdm spacy unidecode langdetect

# Download spaCy language models for EN/DE/FR/IT
!python -m spacy download en_core_web_sm
!python -m spacy download de_core_news_sm
!python -m spacy download fr_core_news_sm
!python -m spacy download it_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 13.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 21.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 111.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 120.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart

## 2. Mount Google Drive <a id="mount-google-drive"></a>

We mount Google Drive so we can load and save our JSON files directly from there.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. One-Time Helpers & Utilities <a id="helpers"></a>

In this section, we define:

- **Global constants** for language detection (`LANGS`)
- **HTML/Markdown** scrubbing regex
- **German Umlaut** mapping
- A **unified** `clean_text` function that can handle any string or list of strings (paragraphs) in a single shot.

In [ ]:
import json, pathlib, re, unicodedata
from tqdm.auto import tqdm
from langdetect import detect
import spacy
import spacy.lang.en, spacy.lang.de, spacy.lang.fr, spacy.lang.it

# Supported language models for spaCy
LANGS = {
    "en": "en_core_web_sm",
    "de": "de_core_news_sm",
    "fr": "fr_core_news_sm",
    "it": "it_core_news_sm",
}

# We'll keep the spaCy pipeline references here if needed later
NLP = {k: None for k in LANGS}

def get_nlp(lang: str):
    """
    Lazy-load spaCy model for the specified language code (en/de/fr/it).
    """
    if lang not in NLP:
        raise ValueError(f"Unsupported language '{lang}'")
    if NLP[lang] is None:
        NLP[lang] = spacy.load(LANGS[lang], disable=["ner", "parser"])
    return NLP[lang]

# Regex patterns for scrubbing
TAG_RE = re.compile(r"</?[^>]+>")
MD_RE  = re.compile(r"^#+\s*", re.M)
WS_RE  = re.compile(r"\s+")

# German Umlaut mapping
UMAP = str.maketrans({
    "ä": "ae", "ö": "oe", "ü": "ue",
    "Ä": "Ae", "Ö": "Oe", "Ü": "Ue", "ß": "ss",
})

def clean_text(input_data, lang: str) -> object:
    """
    A unified text-cleaning function that:
      1) Removes HTML tags
      2) Removes Markdown headings
      3) Normalizes (NFKC)
      4) German-specific Umlaut mapping
      5) Collapses extra whitespace
      6) Converts to lowercase

    It can handle either:
      - A single string (returns a cleaned string)
      - A list of strings (returns a list of cleaned strings)

    Parameters:
        input_data (str | list[str]): The text or list of paragraphs to clean.
        lang (str)                  : The detected or known language code.

    Returns:
        A single cleaned string or a list of cleaned strings, depending on input_data type.
    """

    def _clean_once(text: str) -> str:
        text = TAG_RE.sub(" ", text)
        text = MD_RE.sub(" ", text)
        text = unicodedata.normalize("NFKC", text)
        if lang == "de":
            text = text.translate(UMAP)
        text = WS_RE.sub(" ", text).strip().lower()
        return text

    if isinstance(input_data, str):
        return _clean_once(input_data)
    elif isinstance(input_data, list):
        return [_clean_once(p) for p in input_data]
    else:
        # If somehow it's neither string nor list, just return as-is or raise an error
        return input_data


## 4. Single Processing Function <a id="single-processing-function"></a>

Below, `process_json` does the following in **one** place:

1. **Load** the JSON file.
2. **Detect** the language (or use the existing `"language"` field).
3. **Clean** the main content (`main_content → main_content_clean`).
4. **Clean** the paragraph list (`paragraphs_cleaned → paragraphs_cleaned_1`).
5. **Compute** a cleaned title.
6. **Write** the updated document back to disk.


In [ ]:
def process_json(fp: pathlib.Path):
    """
    Cleans a single JSON file in-place by:
      • Lowercasing + normalizing main_content => main_content_clean
      • Lowercasing + normalizing paragraphs => paragraphs_cleaned_1
      • Generating a cleaned title => title_cleaned
    """
    # 1) Load JSON
    with fp.open("r", encoding="utf-8") as f:
        doc = json.load(f)

    # 2) Determine language (fallback to 'en' if missing)
    raw_main = doc.get("main_content", "")
    lang = (doc.get("language") or detect(raw_main)).split("-")[0]
    if lang not in LANGS:
        # If it's not one of the 4 supported languages, we can skip
        return

    # 3) Clean main content
    doc["main_content_clean"] = clean_text(raw_main, lang)

    # 4) Clean paragraphs
    raw_paragraphs = doc.get("paragraphs_cleaned") or []
    doc["paragraphs_cleaned_1"] = clean_text(raw_paragraphs, lang)

    # 5) Clean or recompute the title
    raw_title = doc.get("title") or ""
    doc["title_cleaned"] = clean_text(raw_title, lang)

    # 6) Write the updated doc back
    with fp.open("w", encoding="utf-8") as fout:
        json.dump(doc, fout, ensure_ascii=False, indent=2)

## 5. Compute Title from Filenames <a id="compute-title-from-filenames"></a>

Many documents have a `filename` but no valid `title`. Below, we show how to auto-generate a more human-friendly `title` from the filename (e.g., `"my-file-name.html"` → `"My File Name"`). Then we validate that every file has **some** title.

In [ ]:
%%time
def filename_to_title(filename: str) -> str:
    """
    Convert a filename like 'some-file-name.html' into a title like 'Some File Name'.
    """
    name = pathlib.Path(filename).name
    if name.lower().endswith('.html'):
        name = name[:-5]  # strip extension
    # Replace hyphens/underscores with spaces
    spaced = name.replace('-', ' ').replace('_', ' ')
    # Title-case
    return spaced.title()

ROOT_DIR = pathlib.Path("/content/drive/MyDrive/GenAI/BSD_advanced_validated")
all_jsons = list(ROOT_DIR.rglob("*.json"))

# 1) Assign or fix up each doc's 'title' field
for js_path in all_jsons:
    with js_path.open('r', encoding='utf-8') as f:
        data = json.load(f)

    fn = data.get('filename', js_path.name)
    data['title'] = filename_to_title(fn)  # Overwrite or set if missing

    # Write back
    with js_path.open('w', encoding='utf-8') as fout:
        json.dump(data, fout, ensure_ascii=False, indent=2)

# 2) Validation: check for missing or empty titles
missing = []
for js_path in all_jsons:
    with js_path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    if not data.get('title'):
        missing.append(js_path.name)

if not missing:
    print("\u2714 All JSON files have a non-empty title.")
else:
    print(f"\u26A0 {len(missing)} files are missing titles:")
    for name in missing:
        print("  -", name)

✔ All JSON files have a non-empty title.
CPU times: user 3.42 s, sys: 1.14 s, total: 4.57 s
Wall time: 1min 24s


## 6. Run the Main Pre-processing Pipeline <a id="run-pipeline"></a>

We now tie everything together: for each `.json` in `ROOT_DIR`, we perform the cleaning steps (`main_content`, paragraphs, and title), ensuring **all** are lowercased and normalized.

In [ ]:
%%time
ROOT_DIR = pathlib.Path("/content/drive/MyDrive/GenAI/BSD_advanced_validated")
all_jsons = list(ROOT_DIR.rglob("*.json"))
print(f"{len(all_jsons)} JSON files found under {ROOT_DIR}")

for fp in tqdm(all_jsons, desc="Full Pre-processing"):
    try:
        process_json(fp)
    except Exception as e:
        print(f"\u26A0  {fp.relative_to(ROOT_DIR)} – {e}")

print("\u2705  Entire corpus processed with unified logic for main_content and paragraphs.")

3544 JSON files found under /content/drive/MyDrive/GenAI/BSD_advanced_validated


Full Pre-processing:   0%|          | 0/3544 [00:00<?, ?it/s]

✅  Entire corpus processed with unified logic for main_content and paragraphs.
CPU times: user 5.99 s, sys: 771 ms, total: 6.76 s
Wall time: 28.3 s


## 7. Sanity Check <a id="sanity-check"></a>

Finally, we pick a single random JSON to confirm everything worked as expected. We’ll show:
- The `language`
- A preview of the `main_content_clean`
- A sample paragraph from `paragraphs_cleaned_1`


In [ ]:
test_file = next((p for p in all_jsons if p.exists()), None)
if test_file:
    with test_file.open('r', encoding='utf-8') as f:
        doc = json.load(f)

    print("\n── Sanity check ───────────────────────────────────────────")
    print("File:              ", test_file.name)
    print("Language:          ", doc.get("language"))
    print("Title (raw):       ", (doc.get("title") or "")[:60], "..." if len(doc.get("title",""))>60 else "")
    print("Title (cleaned):   ", doc.get("title_cleaned", ""))

    mc_clean = doc.get("main_content_clean", "")
    print("\n[main_content_clean] (first 200 chars):")
    print(mc_clean[:200], "..." if len(mc_clean) > 200 else "")

    paragraphs = doc.get("paragraphs_cleaned_1", [])
    if paragraphs:
        print("\n[paragraphs_cleaned_1] - preview of first paragraph:")
        print(paragraphs[0][:200], "..." if len(paragraphs[0]) > 200 else "")
    else:
        print("[paragraphs_cleaned_1]: Not found or empty.")

    print("────────────────────────────────────────────────────────────\n")
else:
    print("No JSON files found for sanity-check.")


── Sanity check ───────────────────────────────────────────
File:               b3b8db4d0d14c3bcf3e9ee1906fb0dd20169d190.json
Language:           de
Title (raw):        Man Muss Sich Die Zeit Gut Einteilen 
Title (cleaned):    man muss sich die zeit gut einteilen

[main_content_clean] (first 200 chars):
es stehen ihnen beratungsstellen fuer (fast) alle situationen im studium zur verfuegung. 

[paragraphs_cleaned_1] - preview of first paragraph:
es stehen ihnen beratungsstellen fuer (fast) alle situationen im studium zur verfuegung. 
────────────────────────────────────────────────────────────



# 8. Chunking (Fixed-Size & Semantic) <a id="chunking"></a>

We process multilingual (de, en, it, fr) news articles that include a `title_cleaned` and `main_content_clean` or `paragraphs_cleaned_1`, applying two chunking methods:

**1. Fixed-size chunking (~512 tokens) with overlap (default 64).**

- Leverages an MPT-7B tokenizer (up to 65k tokens) to avoid truncation errors.

- Title is prepended to `main_content_clean`, ensuring key info in the title is not lost.

**2. Semantic chunking (topic-shift detection).**

- Each paragraph is embedded with a multilingual sentence transformer (`distiluse-base-multilingual-cased-v2`).

- Adjacent paragraphs are merged if their embeddings have **cosine similarity ≥ threshold** (default 0.75).

- After merging, we also enforce a strict **minimum chunk size of 50 tokens**, merging any small chunk with its neighbor until it reaches that threshold.

We then output two lists per record:

- **chunks_fixed_size** (with overlap)

- **chunks_semantic** (based on paragraph merging with a minimum of 50 tokens)

This strategy helps in retrieval-augmented tasks by keeping relevant context from article titles and handling boundary overlaps, while also ensuring that each semantic chunk is large enough to be useful.

The resulting dictionary will then look like this:

```json
// one record/article/html-file
//document_level
{
  "id": "6bfae9b250b03eda1ede9e9eaacf4f04e72bd037_DOC_LEVEL",
  "text": "kleinstmagnete fuer zukuenftige datenspeicher...",
  "metadata": {
    "doc_id": "6bfae9b250b03eda1ede9e9eaacf4f04e72bd037",
    "filename": "kleinstmagnete-fuer-zukuenftige-datenspeicher.html",
    "domain": "ethz.ch",      // always
    "language": "de",         //en, it, fr
    "title": "kleinstmagnete fuer zukuenftige datenspeicher",
    "year": 2017,
    "month": 3,
    "source": "ETH News",     // always
    "doc_named_entities":[{"text": "Atom mit Oberflaeche", "label": "MISC"}, {"text": "Christophe Copéret",  "label": "PER"},  {"text": "ETH Zuerich", "label": "ORG"}, …}],
    "keywords": ["Professor am Laboratorium", "ETH Zuerich",…],
    "text_stats": {
      "char_count": 2450,
      "word_count": 323,
      "paragraph_count": 8}}
}
// fixed_size_chunk
{
  "id": "6bfae9b250b03eda1ede9e9eaacf4f04e72bd037_6bfae9b250b03eda1ede9e9eaacf4f04e72bd037_fixed_0",
  "text": "kleinstmagnete fuer zukuenftige datenspeicher: atom ...",
  "metadata": {
    "doc_id": "6bfae9b250b03eda1ede9e9eaacf4f04e72bd037",
    "filename": "kleinstmagnete-fuer-zukuenftige-datenspeicher.html",
    "domain": "ethz.ch",
    "language": "de",
    "title": "kleinstmagnete fuer zukuenftige datenspeicher",
    "year": 2017,
    "month": 3,
    "source": "ETH News",
    "doc_named_entities":[{"text": "Atom mit Oberflaeche", "label": "MISC"}, {"text": "Christophe Copéret",  "label": "PER"},  {"text": "ETH Zuerich", "label": "ORG"}, …}],
    "keywords": ["Professor am Laboratorium", "ETH Zuerich",…],
    "text_stats": {
      "char_count": 2450,
      "word_count": 323,
      "paragraph_count": 8
    "chunk_id": "6bfae9b250b03eda1ede9e9eaacf4f04e72bd037_fixed_0",
    "chunk_token_count": 512,
    "chunk_named_entities": [{"text": "Atom mit Oberflaeche", "label": "MISC"}, {"text": "Christophe Copéret",  "label": "PER"}],
    "chunk_keywords": ["Professor am Laboratorium", "ETH Zuerich"]}
}
/// till end of length
{"id": "…_fixed_1","..._fixed_2", ...}
// Same for semantic_chunk(token length differs)
{"id": "…_semantic_1","..._semantic_2", ...}
```

## Notebook Setup & Imports

In [ ]:
!pip install --quiet transformers sentence-transformers accelerate spacy yake lingua-language-detector

# For German, English, French, Italian spaCy models:
!python -m spacy download de_core_news_sm
!python -m spacy download en_core_web_sm
!python -m spacy download fr_core_news_sm
!python -m spacy download it_core_news_sm

import re
import json
import pathlib
import logging
from typing import List, Dict, Any

import torch
import numpy as np
import spacy
import yake
import statistics
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer
from lingua import Language, LanguageDetectorBuilder

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

detector = (
    LanguageDetectorBuilder
    .from_languages(Language.ENGLISH, Language.GERMAN, Language.FRENCH, Language.ITALIAN)
    .with_preloaded_language_models()
    .build()
)

spacy_models = {}    # e.g. {"en": spacy_model, "de": spacy_model}
yake_extractors = {} # e.g. {"en": yake_extractor}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.2/96.2 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 117.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

## Helper Functions

In [ ]:
# Lazy-loading spaCy & YAKE & Lingua detection
def get_spacy_model(lang_code: str):
    """
    Lazy-load spaCy for 'en' or 'de' (extend as needed).
    """
    model_map = {"en": "en_core_web_sm", "de": "de_core_news_sm"}
    if lang_code not in model_map:
        return None
    if lang_code in spacy_models:
        return spacy_models[lang_code]
    try:
        nlp = spacy.load(model_map[lang_code])
        spacy_models[lang_code] = nlp
        return nlp
    except Exception as e:
        logging.error(f"SpaCy load error for {lang_code}: {e}")
        return None

def detect_lang(text: str) -> str:
    """
    Use Lingua to detect language. Return ISO639-1 code like 'en', 'de', etc.
    """
    if not text.strip():
        return ""
    try:
        lang = detector.detect_language_of(text)
        return lang.iso_code_639_1.name.lower() if lang else ""
    except Exception as e:
        logging.error(f"Language detection error: {e}")
        return ""

def extract_entities_spacy(text: str, lang_code: str) -> List[dict]:
    """
    Named Entity Recognition with spaCy if model is available for lang_code.
    Returns a list of dicts: {"text": "...", "label": "..."}.
    """
    nlp = get_spacy_model(lang_code)
    if not nlp or not text.strip():
        return []
    doc = nlp(text)
    seen = set()
    ents = []
    for e in doc.ents:
        ent_str = e.text.strip()
        if ent_str not in seen:
            seen.add(ent_str)
            ents.append({"text": ent_str, "label": e.label_})
    return ents

def extract_keywords_yake(text: str, lang_code: str, top_k=10) -> List[str]:
    """
    Keyword extraction with YAKE. If language unsupported, fallback to 'en'.
    """
    if not text.strip():
        return []
    yake_lang = lang_code if lang_code in ["en","de","fr","it"] else "en"

    if yake_lang not in yake_extractors:
        try:
            yake_extractors[yake_lang] = yake.KeywordExtractor(lan=yake_lang, n=3, top=top_k)
        except Exception as e:
            logging.error(f"Failed to init YAKE for {yake_lang}: {e}")
            yake_extractors[yake_lang] = yake.KeywordExtractor(lan="en", n=3, top=top_k)

    extractor = yake_extractors[yake_lang]
    try:
        kw_scored = extractor.extract_keywords(text)
        kw_scored.sort(key=lambda x: x[1])  # sort ascending by score
        return [k for (k, score) in kw_scored[:top_k]]
    except Exception as e:
        logging.error(f"YAKE extract error: {e}")
        return []

# Tokenizer & Sentence-Transformer (for chunking)

try:
    # For large context, e.g. MPT-7B-Storywriter
    model_name = "mosaicml/mpt-7b-storywriter"
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    tokenizer.model_max_length = 65536
    logging.info("Loaded MPT-7B-Storywriter tokenizer (max_length=65536).")
except Exception as e:
    # fallback
    logging.warning(f"Could not load MPT-7B-Storywriter tokenizer. Using default BERT. {e}")
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    tokenizer.model_max_length = 512

try:
    embedding_model = SentenceTransformer("sentence-transformers/distiluse-base-multilingual-cased-v2")
    logging.info("Loaded SentenceTransformer: distiluse-base-multilingual-cased-v2.")
except Exception as e:
    logging.error(f"Could not load distiluse-base-multilingual-cased-v2. {e}")
    embedding_model = None

def add_ner_and_keywords(doc: dict) -> dict:
    """
    Re-run spaCy NER & YAKE on doc-level (main_content_clean) and chunk-level (chunk_text).
    Overwrites doc["named_entities"], doc["keywords"], chunk["named_entities"], chunk["keywords"].
    """
    doc_text = doc.get("main_content_clean", "").strip()
    doc_lang = doc.get("language", "").lower()
    if not doc_lang:
        doc_lang = detect_lang(doc_text)

    # doc-level
    doc["named_entities"] = extract_entities_spacy(doc_text, doc_lang)
    doc["keywords"] = extract_keywords_yake(doc_text, doc_lang)

    # chunk-level: fixed_size
    for chunk in doc.get("chunks_fixed_size", []):
        ctxt = chunk.get("chunk_text", "").strip()
        chunk["named_entities"] = extract_entities_spacy(ctxt, doc_lang)
        chunk["keywords"] = extract_keywords_yake(ctxt, doc_lang)
        chunk["chunk_embedding"] = []  # or store embeddings if you like

    # chunk-level: semantic
    for chunk in doc.get("chunks_semantic", []):
        ctxt = chunk.get("chunk_text", "").strip()
        chunk["named_entities"] = extract_entities_spacy(ctxt, doc_lang)
        chunk["keywords"] = extract_keywords_yake(ctxt, doc_lang)
        chunk["chunk_embedding"] = []  # placeholder

    return doc

def add_ner_and_keywords_all(records: List[dict]) -> List[dict]:
    updated = []
    for r in records:
        updated.append(add_ner_and_keywords(r))
    return updated

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

...

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

## Chunking Functions



In [ ]:
def chunk_fixed_size(
    text: str,
    max_tokens: int = 512,
    chunk_overlap: int = 64
) -> List[str]:
    """
    Splits `text` into overlapping chunks of ~max_tokens tokens, overlapping by chunk_overlap.
    Example: chunk_0 covers tokens [0..511], chunk_1 covers [448..959], etc.
    """
    if chunk_overlap >= max_tokens:
        raise ValueError("Overlap must be smaller than max_tokens.")
    encoded_ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    chunks = []
    start = 0

    while start < len(encoded_ids):
        end = start + max_tokens
        slice_ids = encoded_ids[start:end]
        chunk_text = tokenizer.decode(slice_ids).strip()
        chunks.append(chunk_text)

        start = end - chunk_overlap
        if start < 0:
            start = 0
        if start >= len(encoded_ids):
            break

    return chunks

def cosine_sim(a, b) -> float:
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def chunk_semantic_paragraphs(
    paragraphs: List[str],
    sim_threshold: float = 0.75
) -> List[str]:
    """
    Merges adjacent paragraphs in `paragraphs` if their embeddings
    have similarity >= sim_threshold; else starts a new chunk.
    Returns a list of chunk texts.
    """
    # Clean up empty paragraphs
    paragraphs = [p.strip() for p in paragraphs if p.strip()]
    if not paragraphs:
        return []
    if len(paragraphs) == 1:
        return [paragraphs[0]]

    # Embed each paragraph
    paras_embeds = embedding_model.encode(paragraphs)
    chunks = []
    current_chunk = paragraphs[0]

    for i in range(1, len(paragraphs)):
        sim = cosine_sim(paras_embeds[i-1], paras_embeds[i])
        if sim < sim_threshold:
            # start a new chunk
            chunks.append(current_chunk)
            current_chunk = paragraphs[i]
        else:
            # merge with previous chunk
            # (optionally add a newline, or just a space, up to you)
            current_chunk += "\n" + paragraphs[i]

    # last chunk
    chunks.append(current_chunk)
    return chunks

def merge_short_chunks_strict(chunks: List[str], min_tokens: int = 50) -> List[str]:
    """
    Ensures each output chunk has at least `min_tokens`, if possible, by
    continuing to merge the next chunk into the current buffer *until*
    the buffer has >= min_tokens tokens (or we run out of chunks).

    Example:
      chunks = ["Short chunk (30 tokens)", "Another short chunk (20 tokens)", "Big chunk (200 tokens)", ...]
      -> merges the first two until they exceed 50, then finalizes.

    This can produce large chunks if a short chunk merges with a big one.
    """
    merged = []
    buffer = ""
    buffer_token_count = 0

    for c in chunks:
        c_len = len(c.split())
        if buffer_token_count == 0:
            # Start a new buffer
            buffer = c
            buffer_token_count = c_len
        else:
            # If the current buffer is under min_tokens, keep merging next chunk
            if buffer_token_count < min_tokens:
                buffer += " " + c
                buffer_token_count += c_len
            else:
                # The buffer is already >= min_tokens, finalize it
                merged.append(buffer)
                buffer = c
                buffer_token_count = c_len

    # After the loop ends, flush the buffer if not empty
    if buffer:
        merged.append(buffer)

    return merged

## Processing Each Record

In [ ]:
def process_record(
    record: dict,
    max_tokens: int = 512,
    chunk_overlap: int = 64,
    semantic_sim_threshold: float = 0.75,
    min_tokens_per_chunk: int = 50
) -> dict:
    """
    1) Fixed-size chunking (title_cleaned + main_content_clean).
    2) Semantic chunking on (title_cleaned + paragraphs_cleaned_1).
    3) Refine semantic chunks by merging short chunks (< min_tokens_per_chunk).
    """

    doc_id = record.get("doc_id", "unknown")
    title_str = record.get("title_cleaned", "").strip()
    main_text = record.get("main_content_clean", "").strip()

    # --- (1) Fixed-size chunking ---
    fixed_list = chunk_fixed_size(main_text, max_tokens, chunk_overlap)

    # --- (2) Semantic chunking ---
    paragraphs_cleaned_1 = record.get("paragraphs_cleaned_1", [])
    semantic_list = chunk_semantic_paragraphs(paragraphs_cleaned_1, sim_threshold=semantic_sim_threshold)

    # --- (3) Refine semantic chunks by merging any that are too short ---
    semantic_list_refined = merge_short_chunks_strict(semantic_list, min_tokens=min_tokens_per_chunk)

    # Attach results
    # (A) fixed-size
    record["chunks_fixed_size"] = []
    for i, chunk_txt in enumerate(fixed_list):
        # Prepend the title AFTER chunking
        chunk_text_final = f"{title_str}: {chunk_txt}" if title_str else chunk_txt
        token_count = len(tokenizer(chunk_text_final, add_special_tokens=False)["input_ids"])
        record["chunks_fixed_size"].append({
            "chunk_id": f"{doc_id}_fixed_{i}",
            "chunk_text": chunk_text_final,
            "token_count": token_count
          })

    # (B) semantic + refined
    record["chunks_semantic"] = []
    for idx, chunk_txt in enumerate(semantic_list_refined):
        # Prepend title AFTER the chunking as well
        chunk_text_final = f"{title_str}: {chunk_txt}" if title_str else chunk_txt
        token_count = len(tokenizer(chunk_text_final, add_special_tokens=False)["input_ids"])
        record["chunks_semantic"].append({
            "chunk_id": f"{doc_id}_sem_{idx}",
            "chunk_text": chunk_text_final,
            "token_count": token_count
        })

    return record

## Batch Chunking

In [ ]:
def chunk_all_records(
    records: List[dict],
    max_tokens: int = 512,
    chunk_overlap: int = 64,
    semantic_threshold: float = 0.75,
    min_tokens_per_chunk: int = 50
) -> List[dict]:
    updated = []
    for rec in records:
        updated_rec = process_record(
            rec,
            max_tokens=max_tokens,
            chunk_overlap=chunk_overlap,
            semantic_sim_threshold=semantic_threshold,
            min_tokens_per_chunk=min_tokens_per_chunk
        )
        updated.append(updated_rec)
    return updated

## Saving Updated Data

In [ ]:
def save_updated_data(updated_records: List[dict], out_folder: pathlib.Path):
    out_folder.mkdir(parents=True, exist_ok=True)
    for rec in updated_records:
        doc_id = rec.get("doc_id", "unknown")
        out_path = out_folder / f"{doc_id}.json"
        with open(out_path, "w", encoding="utf-8") as fout:
            json.dump(rec, fout, ensure_ascii=False, indent=2)

## Functions To Compare Fixed Size vs Semantic chunking

In [ ]:
def compute_stats(values: list[int]) -> dict:
    """
    Compute descriptive statistics for a list of integers (token counts or chunk counts).
    Returns count, mean, median, population std dev, min, max.
    """
    if not values:
        return {
            "count": 0, "mean": 0.0, "median": 0.0,
            "std": 0.0, "min": 0, "max": 0
        }
    return {
        "count": len(values),
        "mean": statistics.mean(values),
        "median": statistics.median(values),
        "std": statistics.pstdev(values),
        "min": min(values),
        "max": max(values),
    }

def compare_chunk_stats_advanced(updated_records: list[dict]):
    """
    Presents a refined comparison of fixed-size vs. semantic chunking results,
    including doc-level chunk counts and token-size stats.

    Columns (per doc):
      - DOC ID (truncated)
      - #Fixed, #Sem: number of chunks
      - FixMean, FixMed, FixMin, FixMax, FixStd (token stats for fixed-size)
      - SemMean, SemMed, SemMin, SemMax, SemStd (token stats for semantic)

    At the end, prints:
      1) Doc-level chunk count stats across all docs (min, max, mean, etc.)
      2) Global token-level stats across all docs (fixed-size vs semantic)

    This way, you see not only how large each chunk is, but also how many chunks
    each doc produces in each method.
    """

    header = (
        f"{'DOC ID':<24} "
        f"{'#Fixed':>6} "
        f"{'#Sem':>5} "
        f"{'FixMean':>7} "
        f"{'FixMed':>7} "
        f"{'FixMin':>6} "
        f"{'FixMax':>6} "
        f"{'FixStd':>7} "
        f"{'SemMean':>7} "
        f"{'SemMed':>7} "
        f"{'SemMin':>6} "
        f"{'SemMax':>6} "
        f"{'SemStd':>7}"
    )
    separator = "-" * 118

    print("# Comparison of Fixed-Size vs. Semantic Chunking (Extended)")
    print(header)
    print(separator)

    doc_count = 0

    # For global token stats (across all chunks)
    all_fixed_token_counts = []
    all_sem_token_counts = []

    # For doc-level chunk count stats
    doc_fixed_chunk_counts = []
    doc_sem_chunk_counts = []

    for rec in updated_records:
        doc_id_full = rec.get("doc_id", "unknown")
        # Truncate doc_id if it's very long
        doc_id = (doc_id_full[:21] + "...") if len(doc_id_full) > 24 else doc_id_full

        fixed_chunks = rec.get("chunks_fixed_size", [])
        sem_chunks = rec.get("chunks_semantic", [])

        if not fixed_chunks and not sem_chunks:
            continue  # skip doc if no chunks at all

        # Number of chunks per doc
        n_fixed = len(fixed_chunks)
        n_sem = len(sem_chunks)
        doc_fixed_chunk_counts.append(n_fixed)
        doc_sem_chunk_counts.append(n_sem)

        # Collect token counts
        fixed_token_counts = [fc["token_count"] for fc in fixed_chunks]
        sem_token_counts = [sc["token_count"] for sc in sem_chunks]

        fix_stats = compute_stats(fixed_token_counts)
        sem_stats = compute_stats(sem_token_counts)

        # Print row
        print(
            f"{doc_id:<24}"
            f"{n_fixed:6d}"
            f"{n_sem:5d}"
            f"{fix_stats['mean']:7.1f}"
            f"{fix_stats['median']:7.1f}"
            f"{fix_stats['min']:6d}"
            f"{fix_stats['max']:6d}"
            f"{fix_stats['std']:7.1f}"
            f"{sem_stats['mean']:7.1f}"
            f"{sem_stats['median']:7.1f}"
            f"{sem_stats['min']:6d}"
            f"{sem_stats['max']:6d}"
            f"{sem_stats['std']:7.1f}"
        )

        # Aggregate tokens across entire corpus
        all_fixed_token_counts.extend(fixed_token_counts)
        all_sem_token_counts.extend(sem_token_counts)

        doc_count += 1

    print(separator)
    print(f"Total documents analyzed: {doc_count}\n")

    # --- 1) Doc-level chunk count stats ---
    # How many chunks does each doc have (for fixed-size vs semantic)?
    fix_chunkcount_stats = compute_stats(doc_fixed_chunk_counts)
    sem_chunkcount_stats = compute_stats(doc_sem_chunk_counts)

    print("## Document-Level Chunk Count Stats")
    print("These stats describe how many chunks each document produced.")
    print(f"  [Fixed-size Chunks per Doc]")
    print(f"    #Docs:  {fix_chunkcount_stats['count']}")
    print(f"    Mean:   {fix_chunkcount_stats['mean']:.1f}   (average #chunks per doc)")
    print(f"    Median: {fix_chunkcount_stats['median']:.1f}")
    print(f"    StdDev: {fix_chunkcount_stats['std']:.1f}")
    print(f"    Min:    {fix_chunkcount_stats['min']}")
    print(f"    Max:    {fix_chunkcount_stats['max']}")

    print(f"\n  [Semantic Chunks per Doc]")
    print(f"    #Docs:  {sem_chunkcount_stats['count']}")
    print(f"    Mean:   {sem_chunkcount_stats['mean']:.1f}")
    print(f"    Median: {sem_chunkcount_stats['median']:.1f}")
    print(f"    StdDev: {sem_chunkcount_stats['std']:.1f}")
    print(f"    Min:    {sem_chunkcount_stats['min']}")
    print(f"    Max:    {sem_chunkcount_stats['max']}")

    # --- 2) Token-level stats across all chunks in the corpus ---
    fix_global = compute_stats(all_fixed_token_counts)
    sem_global = compute_stats(all_sem_token_counts)

    explanation = """
Metrics Explained:
  - #Docs:   Number of documents that actually produced chunks.
  - Mean:    Average number of chunks per doc (for chunk counts), or average tokens per chunk (for token stats).
  - Median:  The median (50th percentile) of chunks per doc or token counts.
  - StdDev:  Population standard deviation. Shows how spread out the values are.
  - Min/Max: The smallest/largest value observed.
""".strip()

    print("\n## Global Token-Size Stats (All Chunks Across All Docs)\n")
    print(explanation)
    print("\n  [Fixed-size Chunk Tokens]")
    print(f"    Count:  {fix_global['count']} chunks in total")
    print(f"    Mean:   {fix_global['mean']:.1f} tokens per chunk")
    print(f"    Median: {fix_global['median']:.1f}")
    print(f"    StdDev: {fix_global['std']:.1f}")
    print(f"    Min:    {fix_global['min']}")
    print(f"    Max:    {fix_global['max']}")

    print(f"\n  [Semantic Chunk Tokens]")
    print(f"    Count:  {sem_global['count']} chunks in total")
    print(f"    Mean:   {sem_global['mean']:.1f} tokens per chunk")
    print(f"    Median: {sem_global['median']:.1f}")
    print(f"    StdDev: {sem_global['std']:.1f}")
    print(f"    Min:    {sem_global['min']}")
    print(f"    Max:    {sem_global['max']}")

**Run**:

In [ ]:
import_path = pathlib.Path("/content/drive/MyDrive/GenAI/BSD_advanced_validated")
all_jsons = list(import_path.rglob("*.json"))
print(f"Found {len(all_jsons)} JSON files under {import_path}")

records = []
for js_path in all_jsons:
    with open(js_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        records.append(data)

print(f"Loaded {len(records)} validated news records.")

Found 3544 JSON files under /content/drive/MyDrive/GenAI/BSD_advanced_validated
Loaded 3544 validated news records.


In [ ]:
%%time
# 1) Chunk them
chunked_data  = chunk_all_records(
    records=records,
    max_tokens=512,         # default chunk size (requirement)
    chunk_overlap=64,       # overlap
    semantic_threshold=0.8, # TODO:: Test it out: we can test between τ = 0.75 – 0.95 (for example langchain uses 0.95: https://python.langchain.com/docs/how_to/semantic-chunker/), our text is pretty aligned so I think we can go somewhere between 0.85-0.95.
    min_tokens_per_chunk=50
)

CPU times: user 3min, sys: 2.76 s, total: 3min 3s
Wall time: 2min 48s


We produce both `named_entities` and `keywords` on the document level (again) and at the chunk level, ensuring they’re derived from the same preprocessed text.

In [ ]:
%%time
# 2) Re-run spaCy + YAKE for doc + chunk level
final_data = add_ner_and_keywords_all(chunked_data)

CPU times: user 32min 44s, sys: 6.02 s, total: 32min 50s
Wall time: 32min 44s


Afterwards we verify whether `main_content_clean` and `paragraphs_cleaned_1` remain mostly the same, then compare each chunk’s chunk_text (with its extracted entities/keywords) to the document-level ones for consistency, knowing they won't match exactly but should still align for retrieval.

In [ ]:
record = final_data[0]
example_key = ["doc_id", "title_cleaned", "main_content_clean", "paragraphs_cleaned_1", "chunks_fixed_size", "chunks_semantic"]
subset_dict = {k: record.get(k, "") for k in example_key}

print("Done chunking. Example:\n", json.dumps(subset_dict, indent=2, ensure_ascii=False))

Done chunking. Example:
 {
  "doc_id": "8067f90908726cef40a0e5eec25c9e47b2da0e6c",
  "title_cleaned": "research collection lesen was interessiert",
  "main_content_clean": "pandemic first the most downloaded article was published by the kof, the swiss economic institute at eth zurich, back in june 2021 and is entitled “external pageeconomic analysis:call_madeexternal pageforecast for 2021/2022 – the upturn has arrived, earlier and stronger than expectedcall_made”. download statistics suggest that interest in the article has risen continuously since its publication. focusing on climate research climate change, or more specifically the extreme events it may cause, is not only the topic of the most downloaded record, but is also the subject of the article with the highest altmetric score: - “external pageflood simulation data of a 100-year designed storm in 656catchment areas of switzerlandcall_made” is the name of the record that has already been downloaded 6,790 times since its publicat

**Usage Notes:**

- Overlapping chunking ensures relevant boundary context is preserved between successive chunks.

- The article’s title is prepended so that unique tokens/keywords in the headline are not lost during retrieval.

- Adjust max_tokens, chunk_overlap, semantic_threshold or min_tokens_per_chunk as desired to fine-tune performance.

# Comparison of Fixed-Size vs. Semantic Chunking
The table below shows a per-document comparison between fixed-size and semantic chunking.
For each document, we list:
  - The number of fixed-size chunks and semantic chunks
  - The average token count per chunk in each method
After the per-document rows, we provide an overall summary across all documents.


In [ ]:
compare_chunk_stats_advanced(final_data[-10:]) # specify All or subset.

# Metadata Enrichment & Final JSON Output

- 1) BM25 Indexing
- 2) Vector Similarity Search
- 3) Graph-Based RAG (Building a Knowledge Graph)

We keep one canonical folder `general` containing doc-level+chunk-level JSON. Then each retrieval method (BM25, vector, graph) references those same files as input.

## General Folder `storage/general`

In [ ]:
def create_general_entries(final_data):
    """
    Convert each document record (with doc-level + chunk-level info)
    into a list of general entries.

    We'll produce:
      1) A doc-level entry (using doc['main_content_clean']).
      2) One entry per chunk in 'chunks_fixed_size' (and/or 'chunks_semantic').

    Each entry = {
      "id": ...,
      "text": ...,
      "metadata": {...}
    }
    """

    general_entries = []

    for doc in final_data:
        # ----- Gather doc-level info -----
        doc_id = doc.get("doc_id", "unknown_id")
        # We'll store doc-level metadata
        doc_metadata = {
            "doc_id": doc_id,
            "filename": doc.get("filename", ""),
            "domain": doc.get("domain", ""),
            "language": doc.get("language", ""),
            "title": doc.get("title_cleaned", ""),
            "year": doc.get("year", ""),
            "month": doc.get("month", ""),
            "source": doc.get("source", ""),
            "doc_named_entities": doc.get("named_entities", []),  # doc-wide NER
            "keywords": doc.get("keywords", []),              # doc-wide keywords
            "text_stats": doc.get("text_stats", {}),
        }
        # Gather the doc-level strings
        title_str = doc.get("title_cleaned", "").strip()
        main_str  = doc.get("main_content_clean", "").strip()

        # ----- (A) Create the doc-level general entry -----
        doc_level_text = doc.get("main_content_clean", "")
        doc_level_id = f"{doc_id}_DOC_LEVEL"

        if doc_level_text.strip():
            general_entries.append({
                "id": doc_level_id,
                "text": f"{title_str}: {main_str}" if title_str else main_str,
                "metadata": doc_metadata.copy()
            })

        # ----- (B) Create chunk-level general entries -----
        # We'll demonstrate for 'chunks_fixed_size' (you can do the same for 'chunks_semantic')
        chunks_fixed = doc.get("chunks_fixed_size", [])
        for chunk in chunks_fixed:
            chunk_id = chunk.get("chunk_id", "unknown_chunk_id")
            chunk_text = chunk.get("chunk_text", "")
            if not chunk_text.strip():
                continue

            # Build chunk-level metadata
            chunk_metadata = doc_metadata.copy()  # copy doc-level metadata
            # Now add chunk-specific metadata
            chunk_metadata["chunk_id"] = chunk_id
            chunk_metadata["chunk_token_count"] = chunk.get("token_count", 0)
            chunk_metadata["chunk_named_entities"] = chunk.get("named_entities", [])
            chunk_metadata["chunk_keywords"] = chunk.get("keywords", [])

            # A unique ID for this chunk
            unique_id = f"{doc_id}_{chunk_id}"

            general_entries.append({
                "id": unique_id,
                "text": chunk_text,
                "metadata": chunk_metadata
            })

        # ----- (C) Create chunk-level general entries for chunks_semantic -----
        chunks_sem = doc.get("chunks_semantic", [])
        for chunk in chunks_sem:
            chunk_id = chunk.get("chunk_id", "unknown_chunk_id")
            chunk_text = chunk.get("chunk_text", "")
            if not chunk_text.strip():
                continue

            chunk_metadata = doc_metadata.copy()  # copy doc-level metadata
            chunk_metadata["chunk_id"] = chunk_id
            chunk_metadata["chunk_token_count"] = chunk.get("token_count", 0)
            chunk_metadata["chunk_named_entities"] = chunk.get("named_entities", [])
            chunk_metadata["chunk_keywords"] = chunk.get("keywords", [])

            unique_id = f"{doc_id}_{chunk_id}"

            general_entries.append({
                "id": unique_id,
                "text": chunk_text,
                "metadata": chunk_metadata
            })


    return general_entries

def store_general_entries_in_subfolders(general_entries, base_dir="./storage/general"):
    """
    Given a list of general entries (each is {id, text, metadata}), write them out
    to separate subfolders under the path "storage/general_storage":
       - storage/general_storage/document_level/
       - storage/general_storage/fixed_size_chunk/
       - storage/general_storage/semantic_chunk/

    Each entry is stored in a .json file, named by its 'id' field.

    Example ID patterns:
      - "fffed21a_DOC_LEVEL"
      - "fffed21a_fffed21a_fixed_0"
      - "fffed21a_fffed21a_sem_2"
    """

    base_path = pathlib.Path(base_dir)
    doc_level_dir = base_path / "document_level"
    fixed_dir = base_path / "fixed_size_chunk"
    sem_dir = base_path / "semantic_chunk"

    # Ensure subfolders exist
    doc_level_dir.mkdir(parents=True, exist_ok=True)
    fixed_dir.mkdir(parents=True, exist_ok=True)
    sem_dir.mkdir(parents=True, exist_ok=True)

    for entry in general_entries:
        entry_id = entry["id"]
        # ID pattern
        if entry_id.endswith("_DOC_LEVEL"):
            # doc-level
            out_file = doc_level_dir / f"{entry_id}.json"
        elif "_fixed_" in entry_id:
            # fixed-size chunk
            out_file = fixed_dir / f"{entry_id}.json"
        elif "_sem_" in entry_id:
            # semantic chunk
            out_file = sem_dir / f"{entry_id}.json"
        else:
            # fallback if we find neither pattern
            out_file = doc_level_dir / f"{entry_id}_UNKNOWN.json"

        # Write the JSON
        with open(out_file, "w", encoding="utf-8") as fout:
            json.dump(entry, fout, ensure_ascii=False, indent=2)

    print(f"Done! Wrote {len(general_entries)} general entries into subfolders under '{base_dir}'.")

**Preview**:

In [ ]:
# Now transform to general entries
general_entries = create_general_entries(final_data)

# Let's pretty-print the first few for demonstration:
print("=== GENERAL ENTRIES ===")
for i, entry in enumerate(general_entries[:3]):  # just show first 3
    print(json.dumps(entry, indent=2, ensure_ascii=False))
    print("--------------------------------------------------")

=== GENERAL ENTRIES ===
{
  "id": "8067f90908726cef40a0e5eec25c9e47b2da0e6c_DOC_LEVEL",
  "text": "research collection lesen was interessiert: pandemic first the most downloaded article was published by the kof, the swiss economic institute at eth zurich, back in june 2021 and is entitled “external pageeconomic analysis:call_madeexternal pageforecast for 2021/2022 – the upturn has arrived, earlier and stronger than expectedcall_made”. download statistics suggest that interest in the article has risen continuously since its publication. focusing on climate research climate change, or more specifically the extreme events it may cause, is not only the topic of the most downloaded record, but is also the subject of the article with the highest altmetric score: - “external pageflood simulation data of a 100-year designed storm in 656catchment areas of switzerlandcall_made” is the name of the record that has already been downloaded 6,790 times since its publication. - the paper with an altme

**Run**:

In [ ]:
# Store them in subfolders:
store_general_entries_in_subfolders(general_entries, base_dir="/content/drive/MyDrive/GenAI/storage/general")


Done! Wrote 36084 general entries into subfolders under '/content/drive/MyDrive/GenAI/storage/general'.


# Implementing Multiple Retrieval Strategies

In this phase, we implement and compare different pre-retrieval and retrieval strategies to extract relevant documents from our structured dataset of news articles. The system must handle both English and German content and support advanced metadata usage, such as named entities and keywords, for improved retrieval.

We have **three** chunking variants:

1. **Document-level**: Each JSON record is a *full article*.
2. **Fixed-size**: Each article split into overlapping chunks (e.g., ~512 tokens each).
3. **Semantic**: Articles split by semantic boundaries (topic shifts).

Within each variant, every record includes fields such as `language`, `title`, `year`, `month`, `doc/chunk_named_entities`, `keywords`, etc. This metadata helps with **filtering** and (in GraphRAG) with **graph-based** expansions.

**Core Approaches**
1. **BM25 (Multilingual)**: A baseline using lexical matching in each language.

2. **Dense Retrieval** (Multilingual Embeddings): Captures semantic meaning across languages.

3. **GraphRAG**: Combines semantic search with graph traversal along metadata edges, potentially retrieving more contextually linked documents.

4. **Hybrid** (BM25 + Dense + GraphRAG): Merges results from all three approaches, deduplicates them, and optionally re-ranks or truncates to top-k.

We measure retrieval performance using standard IR metrics (Precision@k, Recall@k, MRR) and compare the strengths of each method. In practice, the best approach can combine lexical, semantic, and graph expansions to ensure high coverage of relevant information.

The next sections detail how we load the data, build each retrieval approach with **LangChain** + **Chroma**, and evaluate them.

## Environment & Installation

We begin by installing necessary libraries. This includes:

- **langchain, chromadb** for building retrieval pipelines with local vector stores.

- **sentence-transformers** to generate multilingual embeddings (for advanced retrieval options).

- **langchain-graph-retriever[chroma]** for a specialized GraphRAG integration with Chroma.

- **nltk** for tokenization/stemming.

- **langdetect** for language detection.

- **transformers** to load models like M2M100 for on-the-fly translation.


In [ ]:
!pip install bs4 docling langchain chromadb sentence-transformers \
             langchain-graph-retriever[chroma] --quiet
!pip install -U langchain-community
!pip install nltk
!pip install langdetect rank_bm25

import os
import json
import pickle
import re
import functools
from typing import List, Dict, Optional, Tuple

import nltk
nltk.download("punkt")      # for English tokenization
nltk.download("punkt_tab")  # needed for German tokenization

import torch
import langchain

from langchain.docstore.document import Document
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.retrievers import BM25Retriever

from google.colab import drive
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.5/166.5 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.9/80.9 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 116.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Mounted at /content/drive


Then we import everything in one place, and detect the best device (CUDA, MPS, or CPU) for running heavier models.

In [ ]:
# Detect the best available device: CUDA → MPS → CPU
if   torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"⮕ Using device: {DEVICE}")
# You can also enable cudnn benchmark for slight speedups on static-size inputs:
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True

# Common kwargs to pass into any HuggingFace model loader or LangChain embedder
MODEL_KWARGS = {
    "device": DEVICE,
    "trust_remote_code": True
}

⮕ Using device: cuda


## Loading the Three Datasets
We have three JSON files:

- document_level.json (each record = entire article)

- fixed_size_chunks.json (each record = chunk with overlap)

- semantic_chunks.json (each record = chunk split by semantic boundaries)

Each entry has:

```json
{
  "id": "...",
  "text": "...",
  "metadata": {
    "doc_id": "...",
    "language": "...",
    "doc_named_entities": [...],
    "keywords": [...],
    ...
  }
}
```

We load them into **LangChain** Document objects via a helper function:

In [ ]:
import pathlib

ROOT_DIR = pathlib.Path("/content/drive/MyDrive/GenAI/storage/general")

def load_documents_from_folder(subfolder: str) -> List[Document]:
    """
    Loads every .json file under ROOT_DIR/subfolder and
    returns a list of LangChain Documents.
    Each JSON file must contain exactly one record:
      {
        "id": "...",       # already doc_id + optional _fixed_n or _sem_n
        "text": "...",
        "metadata": { ... }
      }
    We copy 'id' into metadata['record_id'] so it's globally unique.
    """
    docs: List[Document] = []
    folder_path = ROOT_DIR / subfolder

    for json_path in folder_path.rglob("*.json"):
        with open(json_path, "r", encoding="utf-8") as f:
            record = json.load(f)
        text_content = record.get("text", "")
        md = record.get("metadata", {})
        # We use the file's top-level "id" (e.g. "doc123_fixed_0") as our record_id
        md["record_id"] = record.get("id", "")
        docs.append(Document(page_content=text_content, metadata=md))

    return docs

# loading all three sets:
docs_doclevel = load_documents_from_folder("document_level")
docs_fixed    = load_documents_from_folder("fixed_size_chunk")
docs_semantic = load_documents_from_folder("semantic_chunk")

print(f"Loaded doc-level:         {len(docs_doclevel)} documents")
print(f"Loaded fixed-size chunks: {len(docs_fixed)} documents")
print(f"Loaded semantic chunks:   {len(docs_semantic)} documents")

Loaded doc-level:         3544 documents
Loaded fixed-size chunks: 9341 documents
Loaded semantic chunks:   23199 documents


## Filtering to English–German
Our QA system focuses on English–German queries. Because the requirement is restricted to these languages, we excluded four French/Italian documents after manually verifying that they did not contain essential information for our QA tasks. This streamlines our pipeline, reduces computational overhead, and ensures we retain all content required for accurate English–German question answering.

In [ ]:
from collections import Counter

def filter_doclevel_en_de(
    docs_doclevel: List[Document],
    allowed_langs=("en", "de")
) -> List[Document]:
    """
    Filters out documents from the doc-level dataset whose 'language' metadata is
    not in allowed_langs. Prints summary stats about how many documents were removed
    and returns the filtered list.
    """

    initial_count = len(docs_doclevel)

    filtered_docs = [
        doc for doc in docs_doclevel
        if doc.metadata.get("language", "unknown") in allowed_langs
    ]
    final_count = len(filtered_docs)

    dropped_count = initial_count - final_count
    dropped_percentage = (100.0 * dropped_count / initial_count) if initial_count else 0.0

    print("=== Document-Level Filtering Summary ===")
    print(f"Initial doc-level documents: {initial_count}")
    print(f"Allowed languages: {allowed_langs}")
    print(f"Remaining after filtering: {final_count}")
    print(f"Dropped {dropped_count} documents ({dropped_percentage:.2f}%).\n")

    return filtered_docs

# Example usage on doc-level data
filtered_doclevel = filter_doclevel_en_de(docs_doclevel, allowed_langs=("en", "de"))

# If desired, you can still filter the other chunk sets similarly:
def filter_chunks_en_de(docs_fixed: List[Document], docs_semantic: List[Document], allowed_langs=("en","de")):
    """
    Filters out 'fr'/'it' from your chunk-level datasets as well (fixed-size, semantic).
    Returns the filtered versions of both sets.
    """
    filtered_fixed = [doc for doc in docs_fixed if doc.metadata.get("language", "unknown") in allowed_langs]
    filtered_semantic = [doc for doc in docs_semantic if doc.metadata.get("language", "unknown") in allowed_langs]
    return filtered_fixed, filtered_semantic

filtered_fixed, filtered_semantic = filter_chunks_en_de(docs_fixed, docs_semantic, allowed_langs=("en", "de"))

print(f"Filtered fixed-size chunks:   from {len(docs_fixed)} down to {len(filtered_fixed)}")
print(f"Filtered semantic chunks:     from {len(docs_semantic)} down to {len(filtered_semantic)}")

=== Document-Level Filtering Summary ===
Initial doc-level documents: 3544
Allowed languages: ('en', 'de')
Remaining after filtering: 3540
Dropped 4 documents (0.11%).

Filtered fixed-size chunks:   from 9341 down to 9329
Filtered semantic chunks:     from 23199 down to 23174


We verify how many are removed in each chunk type.


## Text Normalization (Stemming)

To further improve **BM25’s** lexical matching, we normalize text. Specifically, we:

- Lowercase (optionally/done before)
- Remove punctuation with a simple regex.
- Tokenize via nltk.word_tokenize (using German tokenization for “de”).
- Apply SnowballStemmer for English or German.
- Rejoin tokens to create a normalized string.

In [ ]:
from nltk.stem import SnowballStemmer

def normalize_documents(docs: List[Document]) -> List[Document]:
    """
    For each Document in `docs`, we:
      1) Detect language from doc.metadata['language'] (default to 'en').
      2) Remove punctuation via regex.
      3) Tokenize (nltk word_tokenize).
      4) Stem with SnowballStemmer for English or German.
      5) Rejoin tokens into normalized text.
    Returns a new list of Documents with normalized text.
    """
    normalized_docs = []

    # Initialize stemmers
    stemmer_en = SnowballStemmer("english")
    stemmer_de = SnowballStemmer("german")

    for doc in docs:
        lang = doc.metadata.get("language", "en")  # fallback 'en'
        text = doc.page_content.lower()           # ensure it's lowercase

        # 1) Remove punctuation (anything not letter/number/whitespace)
        #    For more selective punctuation removal, adjust the pattern
        text_no_punct = re.sub(r"[^\w\säöüß]", "", text)

        # 2) Tokenize
        tokens = nltk.word_tokenize(text_no_punct, language="german" if lang=="de" else "english")

        # 3) Stem or Lemmatize each token
        #    We'll do a simple SnowballStemmer here
        if lang == "de":
            stemmed_tokens = [stemmer_de.stem(t) for t in tokens]
        else:
            stemmed_tokens = [stemmer_en.stem(t) for t in tokens]

        # 4) Re-join into normalized text
        normalized_text = " ".join(stemmed_tokens)

        # 5) Create a new Document with the normalized content
        new_doc = Document(page_content=normalized_text, metadata=doc.metadata)
        normalized_docs.append(new_doc)

    return normalized_docs

We then create `docs_doclevel_norm`, `docs_fixed_norm`, and `docs_semantic_norm`, which are used for BM25.

In [ ]:
# Normalize text (remove punctuation, tokenize, stem) for each chunk set for BM25
docs_doclevel_norm = normalize_documents(filtered_doclevel)
docs_fixed_norm    = normalize_documents(filtered_fixed)
docs_semantic_norm = normalize_documents(filtered_semantic)

#### Checking Samples
Before building any indexes, we do a quick check of the first few docs from each set (Doc-level, Fixed-size, Semantic) to confirm correct metadata, chunking, and content.

In [ ]:
# quick check for correct format.
for label, docs in [
    ("Doc-level", docs_doclevel_norm),
    ("Fixed-size chunks", docs_fixed_norm),
    ("Semantic chunks", docs_semantic_norm),
]:
    if not docs:
        print(f"{label}: <no documents loaded>")
        continue

    doc = docs[0]
    print(f"--- {label} sample ---")
    print("Text snippet:")
    print(doc.page_content[:200].replace("\n"," "), "...\n")
    print("Metadata:")
    for k, v in doc.metadata.items():
        print(f"  {k}: {v}")
    print("\n")

--- Doc-level sample ---
Text snippet:
stab vppl gut gestartet und bereit fuer noch viel mehr die aufgab des stab doch um welch aufgab kuemmert sich der stab vppl konkret da waer zum ein die allgemein stabsaufgab wie das vorbereit von vern ...

Metadata:
  doc_id: cb2ee71ea4cb29dd2dbff7926160c84308d1e3b7
  filename: stab-vppl-gut-gestartet-und-bereit-fuer-noch-viel-mehr.html
  domain: ethz.ch
  language: de
  title: stab vppl gut gestartet und bereit fuer noch viel mehr
  year: 2021
  month: 4
  source: ETH News
  doc_named_entities: [{'text': 'julia dannath-schuh', 'label': 'PER'}, {'text': 'gremien', 'label': 'LOC'}, {'text': 'vppl', 'label': 'PER'}, {'text': 'schulleitungsbereichs', 'label': 'PER'}, {'text': 'personalentwicklung und lebenslanges', 'label': 'MISC'}, {'text': 'hr', 'label': 'MISC'}, {'text': 'faculty services', 'label': 'ORG'}, {'text': 'einheit faculty services', 'label': 'ORG'}, {'text': 'professuren', 'label': 'MISC'}, {'text': 'eth', 'label': 'ORG'}, {'text': 'dua

Now we have three sets of `Document` objects. Each doc has `.page_content` (the text) and `.metadata` (fields like `language`, `keywords`, `doc_named_entities`).



To store them efficently we will safe the normalized data (`docs_doclevel_norm`, `docs_fixed_norm`, `docs_semantic_norm`)

In [ ]:
# Base path and subfolders
base_path       = "/content/drive/MyDrive/GenAI/storage"
lang_norm_dir   = os.path.join(base_path, "Lang_norm")

doclevel_folder = os.path.join(lang_norm_dir, "document_level")
fixed_folder    = os.path.join(lang_norm_dir, "fixed_size_chunk")
semantic_folder = os.path.join(lang_norm_dir, "semantic_chunk")

# Ensure these directories exist
os.makedirs(doclevel_folder, exist_ok=True)
os.makedirs(fixed_folder,    exist_ok=True)
os.makedirs(semantic_folder, exist_ok=True)

# File paths
doclevel_norm_path  = os.path.join(doclevel_folder, "docs_doclevel_norm.pkl")
fixed_norm_path     = os.path.join(fixed_folder,    "docs_fixed_norm.pkl")
semantic_norm_path  = os.path.join(semantic_folder, "docs_semantic_norm.pkl")

# --- Saving each list of normalized Documents ---
with open(doclevel_norm_path, "wb") as f:
    pickle.dump(docs_doclevel_norm, f)

with open(fixed_norm_path, "wb") as f:
    pickle.dump(docs_fixed_norm, f)

with open(semantic_norm_path, "wb") as f:
    pickle.dump(docs_semantic_norm, f)

print("✅ Saved normalized documents to:")
print(f"  • {doclevel_norm_path}")
print(f"  • {fixed_norm_path}")
print(f"  • {semantic_norm_path}")~


# --- Loading them in a new session if needed ---
# with open(doclevel_norm_path, "rb") as f:
#    docs_doclevel_norm_loaded = pickle.load(f)
# with open(fixed_norm_path, "rb") as f:
#     docs_fixed_norm_loaded = pickle.load(f)
# with open(semantic_norm_path, "rb") as f:
#    docs_semantic_norm_loaded = pickle.load(f)

# print("✅ Loaded normalized docs back into memory.")
# print(f"Doc-level count:  {len(docs_doclevel_norm_loaded)}")
# print(f"Fixed-size count: {len(docs_fixed_norm_loaded)}")
# print(f"Semantic count:   {len(docs_semantic_norm_loaded)}")

✅ Saved normalized documents to:
  • /content/drive/MyDrive/GenAI/storage/Lang_norm/document_level/docs_doclevel_norm.pkl
  • /content/drive/MyDrive/GenAI/storage/Lang_norm/fixed_size_chunk/docs_fixed_norm.pkl
  • /content/drive/MyDrive/GenAI/storage/Lang_norm/semantic_chunk/docs_semantic_norm.pkl


In [ ]:
print("HELLO Fuck my Life")

HELLO Fuck my Life


## BM25 Retrieval (Multilingual Baseline)
BM25 is a classic lexical matching algorithm. To handle both English and German queries, we:

1. **Split** documents by language (en, de).
2. **Build** a separate BM25 index for each language.
3. **Detect query language** and translate the query (if needed) to both languages.
4. **Merge** hits from the English index and German index.
5. **Sort** results by BM25 score, **deduplicate**, and return top-K.

We define a **BilingualBM25** class that encapsulates this logic. For translation, we use M2M100ForConditionalGeneration, although you could choose a lighter bilingual model like `Helsinki-NLP/opus-mt-en-de` if performance or size is a concern.

In [ ]:
import functools
import re
import pickle
from typing import List, Dict, Optional, Tuple

import nltk
from nltk.stem import SnowballStemmer
from langdetect import detect
from transformers import (
    M2M100ForConditionalGeneration,
    M2M100Tokenizer
)
from langchain.docstore.document import Document
from langchain.retrievers import BM25Retriever


###############################################################################
# Fast Bilingual Translator (English <-> German)
###############################################################################
class EnDeTranslator:
    """
    Simple translator for English <-> German queries using a smaller or
    more efficient model if desired, e.g. 'Helsinki-NLP/opus-mt-en-de'.

    For demonstration, we still use facebook/m2m100_418M with:
      - num_beams=1 (greedy decoding)
      - max_new_tokens=128
    """

    def __init__(self, model_name: str = "facebook/m2m100_418M", device: str = "cpu"):
        self.tokenizer = M2M100Tokenizer.from_pretrained(model_name)
        self.model = M2M100ForConditionalGeneration.from_pretrained(model_name).to(device)
        self.device = device

    @functools.lru_cache(maxsize=512)
    def translate(self, text: str, target_lang: str) -> str:
        # Detect source language (fallback to "en" if not recognized)
        src_lang = detect(text)
        if src_lang not in ("en", "de"):
            src_lang = "en"

        # If the source language == target language, no translation needed
        if src_lang == target_lang:
            return text

        self.tokenizer.src_lang = src_lang
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(self.device)

        forced_id = self.tokenizer.get_lang_id(target_lang)
        outputs = self.model.generate(
            **inputs,
            forced_bos_token_id=forced_id,
            num_beams=1,          # greedy for speed
            max_new_tokens=128
        )
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)


###############################################################################
# BilingualBM25 for English–German
###############################################################################
class BilingualBM25:
    """
    Builds a BM25Retriever for English and one for German.
    Does cross-lingual search by translating the query if needed.
    Returns merged & deduplicated results.

    If 'translator' is None, *no translation* will occur, meaning:
      - Query is used as-is on both English and German indexes.
      - This is only recommended if your queries/documents are mostly
        in the same language, or if you attach a translator after loading.
    """

    def __init__(
        self,
        docs: List[Document],
        translator: Optional[EnDeTranslator] = None,
        bm25_k1: float = 1.2,
        bm25_b: float = 0.75,
    ):
        # Translator can be None (for easy pickling or if no cross-lingual is required)
        self.translator = translator
        self.k1, self.b = bm25_k1, bm25_b

        # Bucket docs by language ("en" or "de")
        buckets: Dict[str, List[Document]] = {"en": [], "de": []}
        for doc in docs:
            lang = doc.metadata.get("language", "en")
            if lang in ("en", "de"):
                buckets[lang].append(doc)

        # Build a separate BM25 retriever for each language
        self.retrievers: Dict[str, BM25Retriever] = {}
        for lang in ("en", "de"):
            if buckets[lang]:
                self.retrievers[lang] = BM25Retriever.from_documents(
                    buckets[lang], k1=self.k1, b=self.b
                )

    def detect_language(self, query: str) -> str:
        """Detect query language (fallback to 'en' if detection fails or is not 'en'/'de')."""
        try:
            lang = detect(query)
            if lang not in ("en", "de"):
                lang = "en"
        except:
            lang = "en"
        return lang

    def search(
        self,
        query: str,
        top_k: int = 5,
        metadata_filter: Optional[dict] = None
    ) -> List[Document]:
        """
        1) Detect language of query.
        2) If translator is present and the language of the index differs from the query,
           translate query accordingly.
        3) Retrieve from both English & German BM25 indexes, combine results.
        4) Sort by BM25 score, deduplicate, return top_k.
        """
        src_lang = self.detect_language(query)
        all_hits: List[Tuple[Document, float]] = []

        for lang, retriever in self.retrievers.items():
            # If we have a translator, and the language differs, we translate.
            if self.translator is not None and lang != src_lang:
                q = self.translator.translate(query, lang)
            else:
                q = query

            # Retrieve top_k hits from BM25
            hits = retriever.get_relevant_documents(
                q, k=top_k, filter=metadata_filter
            )
            for doc in hits:
                score = doc.metadata.get("score", 0.0)
                all_hits.append((doc, score))

        # Sort by BM25 score (descending) and deduplicate by 'record_id'
        seen = set()
        unique_hits = []
        for doc, score in sorted(all_hits, key=lambda x: x[1], reverse=True):
            rid = doc.metadata.get("record_id")
            if rid not in seen:
                seen.add(rid)
                unique_hits.append(doc)
            if len(unique_hits) >= top_k:
                break

        return unique_hits

### Building & Saving BM25 Indexes

In [ ]:
# Define the base folder and subfolders
base_folder     = "/content/drive/MyDrive/GenAI/storage/retrival"
doclevel_folder = os.path.join(base_folder, "document_level")
fixedsize_folder = os.path.join(base_folder, "fixed_size_chunk")
semantic_folder = os.path.join(base_folder, "semantic_chunk")

# Create directories if they don't exist
os.makedirs(doclevel_folder, exist_ok=True)
os.makedirs(fixedsize_folder, exist_ok=True)
os.makedirs(semantic_folder, exist_ok=True)

We next build three BM25 indexes—one for each chunk variant—using the normalized documents:

In [ ]:
# 1) Build & save the document-level index
print("Building BM25 index for Document-Level...")
bm25_doc = BilingualBM25(docs=docs_doclevel_norm, translator=EnDeTranslator(device=DEVICE))
bm25_doc.translator = None
doc_pickle_path = os.path.join(doclevel_folder, "bm25_retriever.pkl")
with open(doc_pickle_path, "wb") as f:
    pickle.dump(bm25_doc, f)
print(f"Document-level index saved to {doc_pickle_path}")

# 2) Build & save the fixed-size chunks index
print("\nBuilding BM25 index for Fixed-Size Chunks...")
bm25_fixed = BilingualBM25(docs=docs_fixed_norm, translator=EnDeTranslator(device=DEVICE))
bm25_fixed.translator = None
fixed_pickle_path = os.path.join(fixedsize_folder, "bm25_retriever.pkl")
with open(fixed_pickle_path, "wb") as f:
    pickle.dump(bm25_fixed, f)
print(f"Fixed-size chunks index saved to {fixed_pickle_path}")

# 3) Build & save the semantic chunks index
print("\nBuilding BM25 index for Semantic Chunks...")
bm25_sem = BilingualBM25(docs=docs_semantic_norm, translator=EnDeTranslator(device=DEVICE))
bm25_sem.translator = None
sem_pickle_path = os.path.join(semantic_folder, "bm25_retriever.pkl")
with open(sem_pickle_path, "wb") as f:
    pickle.dump(bm25_sem, f)
print(f"Semantic chunks index saved to {sem_pickle_path}")

Building BM25 index for Document-Level...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/3.71M [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

Document-level index saved to /content/drive/MyDrive/GenAI/storage/retrival/document_level/bm25_retriever.pkl

Building BM25 index for Fixed-Size Chunks...
Fixed-size chunks index saved to /content/drive/MyDrive/GenAI/storage/retrival/fixed_size_chunk/bm25_retriever.pkl

Building BM25 index for Semantic Chunks...
Semantic chunks index saved to /content/drive/MyDrive/GenAI/storage/retrival/semantic_chunk/bm25_retriever.pkl


Each index is pickled to Google Drive under subfolders:
- `/content/drive/MyDrive/GenAI/storage/retrival/document_level/bm25_retriever.pkl`
- `/content/drive/MyDrive/GenAI/storage/retrival/fixed_size_chunk/bm25_retriever.pkl`
- `/content/drive/MyDrive/GenAI/storage/retrival/semantic_chunk/bm25_retriever.pkl`

### Testing the Document-Level Index
Finally, we (re-) load the stored bm25_retriever.pkl for doc-level and run a sample query in both English and German:

In [ ]:
# reload the essentials:
!pip install langchain chromadb sentence-transformers \
             nltk langdetect rank_bm25 transformers

import nltk
nltk.download("punkt")      # for English tokenization
nltk.download("punkt_tab")  # needed for German tokenization

from google.colab import drive
drive.mount('/content/drive')


import pathlib
import os
import functools
import pickle
from typing import List, Dict, Optional, Tuple

from langdetect import detect
from transformers import (
    M2M100ForConditionalGeneration,
    M2M100Tokenizer
)
from langchain.docstore.document import Document
from langchain.retrievers import BM25Retriever

###############################################################################
# Fast Bilingual Translator (English <-> German)
###############################################################################
class EnDeTranslator:
    """
    Simple translator for English <-> German queries using a smaller or
    more efficient model if desired, e.g. 'Helsinki-NLP/opus-mt-en-de'.

    For demonstration, we still use facebook/m2m100_418M with:
      - num_beams=1 (greedy decoding)
      - max_new_tokens=128
    """

    def __init__(self, model_name: str = "facebook/m2m100_418M", device: str = "cpu"):
        self.tokenizer = M2M100Tokenizer.from_pretrained(model_name)
        self.model = M2M100ForConditionalGeneration.from_pretrained(model_name).to(device)
        self.device = device

    @functools.lru_cache(maxsize=512)
    def translate(self, text: str, target_lang: str) -> str:
        # Detect source language (fallback to "en" if not recognized)
        src_lang = detect(text)
        if src_lang not in ("en", "de"):
            src_lang = "en"

        # If the source language == target language, no translation needed
        if src_lang == target_lang:
            return text

        self.tokenizer.src_lang = src_lang
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(self.device)

        forced_id = self.tokenizer.get_lang_id(target_lang)
        outputs = self.model.generate(
            **inputs,
            forced_bos_token_id=forced_id,
            num_beams=1,          # greedy for speed
            max_new_tokens=128
        )
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)


###############################################################################
# BilingualBM25 for English–German
###############################################################################
class BilingualBM25:
    """
    Builds a BM25Retriever for English and one for German.
    Does cross-lingual search by translating the query if needed.
    Returns merged & deduplicated results.

    If 'translator' is None, *no translation* will occur, meaning:
      - Query is used as-is on both English and German indexes.
      - This is only recommended if your queries/documents are mostly
        in the same language, or if you attach a translator after loading.
    """

    def __init__(
        self,
        docs: List[Document],
        translator: Optional[EnDeTranslator] = None,
        bm25_k1: float = 1.2,
        bm25_b: float = 0.75,
    ):
        # Translator can be None (for easy pickling or if no cross-lingual is required)
        self.translator = translator
        self.k1, self.b = bm25_k1, bm25_b

        # Bucket docs by language ("en" or "de")
        buckets: Dict[str, List[Document]] = {"en": [], "de": []}
        for doc in docs:
            lang = doc.metadata.get("language", "en")
            if lang in ("en", "de"):
                buckets[lang].append(doc)

        # Build a separate BM25 retriever for each language
        self.retrievers: Dict[str, BM25Retriever] = {}
        for lang in ("en", "de"):
            if buckets[lang]:
                self.retrievers[lang] = BM25Retriever.from_documents(
                    buckets[lang], k1=self.k1, b=self.b
                )

    def detect_language(self, query: str) -> str:
        """Detect query language (fallback to 'en' if detection fails or is not 'en'/'de')."""
        try:
            lang = detect(query)
            if lang not in ("en", "de"):
                lang = "en"
        except:
            lang = "en"
        return lang

    def search(
        self,
        query: str,
        top_k: int = 5,
        metadata_filter: Optional[dict] = None
    ) -> List[Document]:
        """
        1) Detect language of query.
        2) If translator is present and the language of the index differs from the query,
           translate query accordingly.
        3) Retrieve from both English & German BM25 indexes, combine results.
        4) Sort by BM25 score, deduplicate, return top_k.
        """
        src_lang = self.detect_language(query)
        all_hits: List[Tuple[Document, float]] = []

        for lang, retriever in self.retrievers.items():
            # If we have a translator, and the language differs, we translate.
            if self.translator is not None and lang != src_lang:
                q = self.translator.translate(query, lang)
            else:
                q = query

            # Retrieve top_k hits from BM25
            hits = retriever.get_relevant_documents(
                q, k=top_k, filter=metadata_filter
            )
            for doc in hits:
                score = doc.metadata.get("score", 0.0)
                all_hits.append((doc, score))

        # Sort by BM25 score (descending) and deduplicate by 'record_id'
        seen = set()
        unique_hits = []
        for doc, score in sorted(all_hits, key=lambda x: x[1], reverse=True):
            rid = doc.metadata.get("record_id")
            if rid not in seen:
                seen.add(rid)
                unique_hits.append(doc)
            if len(unique_hits) >= top_k:
                break

        return unique_hits

We inspect the top matches’ language, titles, and text snippets to see if it answers the question.

In [ ]:
# 1) Load the classes above (EnDeTranslator, BilingualBM25)
bm25_doc = BilingualBM25(docs_doclevel_norm, translator=EnDeTranslator(device="cpu"))

# 2) Then load your pickled object document-level index or "fixed_size_chunk", "semantic_chunk"
doclevel_folder = "/content/drive/MyDrive/GenAI/storage/retrival/document_level" # change to: "fixed_size_chunk", "semantic_chunk"
doc_pickle_path = os.path.join(doclevel_folder, "bm25_retriever.pkl")

with open(doc_pickle_path, "rb") as f:
    bm25_doc = pickle.load(f)

# 3) Reattach a translator if you want cross-lingual queries
bm25_doc.translator = EnDeTranslator(device="cpu")

# 4) Run a test query in English or German
test_queries = [
    "Who were the rectors of ETH between 2017 and 2022?",
    "Wer waren die Rekoren der ETH zwischen 2017 und 2022?"
]

for query in test_queries:
    print(f"\n>>> Query: {query}")
    hits = bm25_doc.search(query, top_k=5)
    for i, doc in enumerate(hits, start=1):
        print(f" {i}. Language: {doc.metadata.get('language')} | Title: {doc.metadata.get('title')}\n Text: {doc.page_content}")


>>> Query: Who were the rectors of ETH between 2017 and 2022?


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:679: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(


 1. Language: en | Title: what previous bird flu outbreaks teach us
 Text: what previous bird flu outbreak teach us the bird flu epidem in china from 2013 to 2017 show that pathogen can circul in poultri farm for sever month befor be detect virus spread quick at live poultri market the studi author suggest to continu monitor the anim health there are mani differ bird flu virus besid the subtyp h5n1 which has been spread in the european wild bird popul for sever year and pose a threat to local poultri farm there is also for instanc subtyp h7n9 this one caus poultri outbreak in china from 2013 to 2017 and also infect human who had close contact with live poultri a total of 616 peopl in china were report to have die from an infect with this subtyp expert are track how the differ bird flu virus are develop with both h7n9 and other subtyp there is a risk that mutat in their genom could allow for humantohuman transmiss rais the threat of a pandem that whi clair guinat a former postdoc in eth

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:679: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(


 1. Language: en | Title: what previous bird flu outbreaks teach us
 Text: what previous bird flu outbreak teach us the bird flu epidem in china from 2013 to 2017 show that pathogen can circul in poultri farm for sever month befor be detect virus spread quick at live poultri market the studi author suggest to continu monitor the anim health there are mani differ bird flu virus besid the subtyp h5n1 which has been spread in the european wild bird popul for sever year and pose a threat to local poultri farm there is also for instanc subtyp h7n9 this one caus poultri outbreak in china from 2013 to 2017 and also infect human who had close contact with live poultri a total of 616 peopl in china were report to have die from an infect with this subtyp expert are track how the differ bird flu virus are develop with both h7n9 and other subtyp there is a risk that mutat in their genom could allow for humantohuman transmiss rais the threat of a pandem that whi clair guinat a former postdoc in eth